# Task A Run 16 -- the funnel: screen everything, confirm the leaders

This replaces Runs 12 to 15 as separate notebooks. They test one factor each against their
own control, cost about 30 hours in total, and still never compare the factors on one
table.

## Why a funnel rather than eight submissions

Eight full-data submissions cannot answer the question. The CodaBench validation set is 806
rows and one score there carries about **1.5 points of standard deviation**, while most of
these ideas are worth one or two. Most gaps would come back inside noise.

Eight five-fold runs would answer it, at ~160 minutes each.

So: screen every arm on the cheap split first, promote only the leaders. Task B ran exactly
this funnel and recorded the result — *"for all four promoted arms the five-fold order
matched the holdout order exactly, while the level dropped by roughly 0.022 in three of the
four cases."* The holdout **ranks** correctly and **reads high**, so stage 1 orders arms and
never reports a number.

| stage | what it does | cost per arm |
|---|---|---|
| 1, screen | every arm on the fixed 15% holdout, 960 rows | ~33 min |
| 2, confirm | the leaders plus the control on five folds, 6,401 rows | ~160 min |
| 3, ship | fit the blend weight and threshold on the winner's OOF, refit on all rows | ~35 min |

Stage 3 is Run 11, which already exists as `11_reinit1_full_data.ipynb`.

## The arms

Every arm changes the **MuRIL component only**. The TF-IDF/SVM half of the blend is fixed,
because nothing queued touches it. Arms are component-level because that is the dependency
order: a blend weight depends on how strong its components are, so the component has to be
settled first, or the weight is invalidated by the next change.

| arm | the change | why |
|---|---|---|
| `control` | one reinitialized layer, 6 epochs, stock MuRIL | the reference everything is measured against |
| `reinit2` | two layers | Run 9's inherited setting; Task B found one beat none by +3.0 and one beat two by +0.5, inside noise |
| `tapt` | TAPT checkpoint | the largest measured effect in the project, +2.9 on Task B |
| `ext_mix` | external rows appended to training | the external corpus carries exactly `Hate`/`Non-Hate`; +0.0030 on the TF-IDF floor |
| `ext_stage` | fine-tune on external first, then Task A | lets clean labels overwrite the external annotation's boundary |
| `epochs10` | 10 epochs | Task A's own logs show validation F1 still rising at epoch 6 |
| `large` | MuRIL-large, `--no-fgm`, `--lr 1e-5` | 24 layers; 6,401 rows support it better than Task B's 3,143 |

Deliberately absent, all measured and dead — see [`docs/IDEAS.md`](../../docs/IDEAS.md):
gazetteer and profanity features, stemming, stopword removal, focal loss, class weighting,
frozen embeddings.

## Two safeguards

**Collapse detection.** Every arm's predicted positive rate is checked against Task A's
0.491 prior. An arm calling one class on over 90% of rows has collapsed and is not
promoted, which is what mDeBERTa did on Task B at a learning rate that suited the base
model.

**Resumability.** Every stage skips work whose output already exists, and `RESULTS.md` is
rewritten after each arm. A session that dies partway can be re-run and continues.

## Runtime

Stage 1 is about **6 hours** including the TAPT build. Stage 2 adds roughly 160 minutes per
promoted arm plus the control, so plan a second session for it.

Set **Accelerator** to `GPU T4 x2` or `GPU P100` and **Internet** on, then
**Save Version -> Save & Run All**.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import numpy as np
import pandas as pd
import torch
from hastika.common.preprocessing import dedupe_index

assert torch.cuda.is_available(), "no GPU -- select a CUDA-enabled runtime"
print("gpu:", torch.cuda.get_device_name(0))
train = pd.read_csv("data/raw/binary_train.csv")
keep = dedupe_index(train["Comment"].tolist(), train["Label"].tolist(), "task A")
print(f"raw labelled rows: {len(train)}; deduplicated rows used for fitting: {len(keep)}")
assert len(keep) == 6401, len(keep)

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Run the funnel

Six arms, about **5.2 hours**. `tapt` is excluded because Run 12 is already answering it
at five folds, which is better evidence than a holdout screen; every other arm here has
never been run on Task A.

`--stage screen` now, `--stage confirm` in a later session once you know which arms lead.
`RESULTS.md` is rewritten after every arm, so a session that dies partway still leaves a
usable ranking.

In [ ]:
import time
t0 = time.time()
# `tapt` is deliberately absent: Run 12 is already answering it at five folds, which
# beats a holdout screen. Every other arm here has never been run on Task A at all.
# No TAPT checkpoint is built, so this starts immediately.
ARMS = ["control", "reinit2", "ext_mix", "ext_stage", "epochs10", "large"]

run([sys.executable, "-u", "experiments/task_a/funnel.py",
     "--budget-hours", "10.5", "--reserve-min", "20",
     "--stage", "screen",
     "--arms", *ARMS,
     "--promote", "2",
     "--out", "/kaggle/working"],
    log="artifacts/logs/task_a_funnel.log")
print(f"\nelapsed {(time.time() - t0) / 3600:.2f} h")

## 2. Read the table

`holdout` orders the arms. `5-fold` is the number to report, and is only filled in for
arms that reached stage 2. `hate_rate` is the collapse check: Task A's prior is 0.491.

A holdout gap under about 0.013 is inside that split's noise. A five-fold gap under about
0.006 is inside its own.

In [ ]:
RESULTS = pathlib.Path("/kaggle/working/RESULTS.md")
print(RESULTS.read_text() if RESULTS.exists() else "RESULTS.md missing -- check the log")

## 3. Preserve outputs

The `oof_probs.npy` files from any stage-2 arm are the durable product: they make the
stage-3 weight and threshold fit free, and every future blend idea after that.

In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_funnel_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for d in sorted(pathlib.Path("artifacts/runs").glob("[af]_*")):
    for name in ["oof_probs.npy", "holdout_probs.npy", "test_probs.npy"]:
        p = d / name
        if p.exists():
            shutil.copy2(p, OUT / f"{d.name}_{name}")
for log in pathlib.Path("artifacts/logs").glob("[af]_*.log"):
    shutil.copy2(log, OUT / log.name)
for name in ["RESULTS.md", "results.json"]:
    p = pathlib.Path("/kaggle/working") / name
    if p.exists():
        shutil.copy2(p, OUT / name)
print(sorted(x.name for x in OUT.iterdir()))

## 4. What to do with the result

Record the whole table in `docs/EXPERIMENTS.md`, **including the arms that lost**. A
negative result written down is what stops the same idea being retried in three weeks; that
is what `docs/IDEAS.md` exists for.

Then:

1. Run stage 2 in a second session: `--stage confirm --promote 2`.
2. Take the winning arm into Run 11, which fits the blend weight and threshold on its
   out-of-fold probabilities and refits both components on all 6,401 rows.
3. Submit that one ZIP.

By the end you will have one table saying what helps and what does not, measured on 6,401
rows rather than inferred from a handful of 806-row leaderboard scores — and exactly one
submission spent on the answer rather than eight spent failing to find it.